# Critics vs Players: Should You Send Review Copies?

**Business Question:** As a game publisher, is it worth sending review copies to critics? Does it drive sales and engagement?

**User Persona:** Thomas, game publisher about to launch a new PC (Windows) game on steam after years of development.

---

## ⚠️ **Important Disclaimer: Toy Project**

> **This is a data science toy project for educational purposes only.**  
> 
> **Limitations:**
> - **Single critic source**: Only uses IGN reviews (not representative of all gaming critics)
> - **Platform limited**: Steam data for Windows PC games only (excludes consoles, Mac, Linux)
> - **Sample bias**: Dataset may not represent the full gaming market
> - **Not production-ready**: Do not use for actual business decisions without additional research
> 
> For real business decisions, consult multiple review aggregators (Metacritic, OpenCritic), cross-platform data, and professional market research.

---

## Executive Summary
This analysis examines the relationship between critic scores, sales (owners), player engagement, and pricing to determine the ROI of critic reviews.

## Data Loading & Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
df = pd.read_csv('output/output.csv')

# Display basic info
print(f"Total games: {len(df)}")
print(f"Date range: {df['release_date'].min()} to {df['release_date'].max()}")
df.head()

Total games: 1106
Date range: 2003-01-29 to 2016-09-22


,title,release_date,genre,critic_score,critic_score_phrase,developer,publisher,required_age,average_playtime,median_playtime,owners,price,main_story,main_extra,completionist,all_styles
0,1001 Spikes,2014-06-08,Platformer,8.0,Great,"Nicalis, Inc.","Nicalis, Inc.",0,214,215,75000,9.99,7.14,14.37,16.91,13.38
1,140,2013-10-16,Platformer,8.0,Great,Carlsen Games,Carlsen Games,0,205,274,350000,3.99,1.20,2.23,5.26,1.56
2,1979 Revolution,2016-04-21,"Action,Adventure",8.0,Great,iNK Stories;N-Fusion Interactive,iNK Stories,0,0,0,75000,4.79,2.10,2.85,5.88,2.66
3,7 Wonders: Treasures of Seven,2008-11-06,Puzzle,5.9,Mediocre,MumboJumbo,MumboJumbo,0,0,0,35000,6.99,7.10,7.42,9.00,7.43
4,A Bird Story,2014-11-26,RPG,8.8,Great,Freebird Games,Freebird Games,0,257,476,350000,2.79,1.27,1.39,1.41,1.34


## Data Cleaning & Feature Engineering

Prepare metrics for analysis:
- **Engagement ratio**: How much longer players play vs. expected completion time
- **Revenue proxy**: Estimated revenue (price × owners midpoint)
- **Completion rate**: Percentage who finish the main story
- **Primary genre**: Extract main genre from multi-genre strings

In [2]:
# Convert release_date to datetime
df['release_date'] = pd.to_datetime(df['release_date'])

# Extract owners midpoint (assuming "75000" means 0-150k range, we'll use the value as midpoint)
df['owners_midpoint'] = df['owners']

# Calculate engagement ratio (how much more than expected completion time)
df['engagement_ratio'] = np.where(
    (df['all_styles'] > 0) & (df['median_playtime'] > 0),
    df['median_playtime'] / df['all_styles'],
    np.nan
)

# Calculate revenue proxy
df['revenue_proxy'] = df['price'] * df['owners_midpoint']

# Calculate completion rate
df['completion_rate'] = np.where(
    df['median_playtime'] > 0,
    (df['main_story'] / df['median_playtime']) * 60,  # Convert to percentage
    np.nan
)

# Extract primary genre
df['primary_genre'] = df['genre'].str.split(',').str[0]

# Filter out games with missing critical data
df_analysis = df[
    (df['critic_score'].notna()) & 
    (df['owners'] > 0) & 
    (df['price'] >= 0)
].copy()

print(f"Games for analysis: {len(df_analysis)}")
print(f"Critic score range: {df_analysis['critic_score'].min():.1f} - {df_analysis['critic_score'].max():.1f}")
print(f"Price range: ${df_analysis['price'].min():.2f} - ${df_analysis['price'].max():.2f}")
print(f"\nTop genres:")
print(df_analysis['primary_genre'].value_counts().head(10))

Games for analysis: 1106
Critic score range: 1.5 - 10.0
Price range: $0.00 - $49.99

Top genres:
primary_genre
Action        255
Strategy      209
Shooter       195
Adventure     133
RPG            96
Puzzle         49
Simulation     48
Platformer     35
Racing         32
Sports         22
Name: count, dtype: int64


---

# Key Analysis: Does Critic Score Drive Success?

## 1. Critic Score → Sales Correlation

**Question:** Do higher critic scores lead to more sales?

**What to look for:** 
- Positive correlation = Critics drive sales ✅
- Flat trend = Critics don't matter ❌
- Strong R² = Predictable relationship

In [3]:
from scipy import stats

# Calculate correlation
corr, p_value = stats.pearsonr(df_analysis['critic_score'], df_analysis['owners_midpoint'])

# Create scatter plot with trend line
fig = px.scatter(
    df_analysis,
    x='critic_score',
    y='owners_midpoint',
    color='primary_genre',
    size='price',
    hover_data=['title', 'price', 'owners_midpoint'],
    title=f'📈 Critic Score vs Sales (Owners)<br><sub>Correlation: {corr:.3f} (p={p_value:.4f})</sub>',
    labels={'critic_score': 'Critic Score', 'owners_midpoint': 'Number of Owners'},
    trendline='ols',
    trendline_scope='overall',
    height=600
)

fig.update_yaxes(type='log', title='Owners (log scale)')
fig.update_layout(
    showlegend=True,
    legend=dict(title='Primary Genre', orientation='v', x=1.02, y=1)
)

fig.show()

print(f"\n💡 Key Insight: Correlation = {corr:.3f}")
if corr > 0.3:
    print("✅ Strong positive relationship! Better reviews = More sales")
elif corr > 0.1:
    print("⚠️ Weak positive relationship. Reviews help but aren't decisive")
else:
    print("❌ No meaningful relationship. Reviews don't predict sales")


💡 Key Insight: Correlation = 0.130
⚠️ Weak positive relationship. Reviews help but aren't decisive


## 2. Critic Score → Player Engagement

**Question:** Do well-reviewed games keep players engaged longer?

**What to look for:**
- Engagement ratio > 1 = Players exceed expected playtime
- Positive correlation = Critics identify engaging games ✅
- This drives word-of-mouth marketing

In [4]:
# Filter data with valid engagement metrics
df_engagement = df_analysis[
    (df_analysis['engagement_ratio'].notna()) & 
    (df_analysis['engagement_ratio'] > 0) &
    (df_analysis['engagement_ratio'] < 10)  # Remove extreme outliers
].copy()

# Calculate correlation
corr_eng, p_value_eng = stats.pearsonr(df_engagement['critic_score'], df_engagement['engagement_ratio'])

# Create scatter plot
fig = px.scatter(
    df_engagement,
    x='critic_score',
    y='engagement_ratio',
    color='primary_genre',
    size='owners_midpoint',
    hover_data=['title', 'median_playtime', 'all_styles'],
    title=f'🎯 Critic Score vs Player Engagement<br><sub>Correlation: {corr_eng:.3f} (p={p_value_eng:.4f})</sub>',
    labels={
        'critic_score': 'Critic Score',
        'engagement_ratio': 'Engagement Ratio (Playtime / Expected Time)'
    },
    trendline='ols',
    trendline_scope='overall',
    height=600
)

fig.add_hline(y=1, line_dash="dash", line_color="red", 
              annotation_text="1.0 = Players match expected playtime")

fig.update_layout(
    showlegend=True,
    legend=dict(title='Primary Genre', orientation='v', x=1.02, y=1)
)

fig.show()

print(f"\n💡 Key Insight: Correlation = {corr_eng:.3f}")
if corr_eng > 0.3:
    print("✅ Strong relationship! Better reviews = More engaged players")
elif corr_eng > 0.1:
    print("⚠️ Weak relationship. Reviews somewhat predict engagement")
else:
    print("❌ No clear relationship between reviews and engagement")


💡 Key Insight: Correlation = 0.053
❌ No clear relationship between reviews and engagement


## 3. ROI Analysis: Revenue Potential by Critic Score

**Question:** Do critic scores translate to actual revenue?

**What to look for:**
- High score/High revenue quadrant = Proven success formula ✅
- Revenue increases with score = Reviews drive profits 💰
- Outliers = Marketing or franchise power

In [6]:
# Filter valid revenue data
df_revenue = df_analysis[df_analysis['revenue_proxy'] > 0].copy()

# Calculate correlation
corr_rev, p_value_rev = stats.pearsonr(df_revenue['critic_score'], df_revenue['revenue_proxy'])

# Add quadrants
median_score = df_revenue['critic_score'].median()
median_revenue = df_revenue['revenue_proxy'].median()

# Create scatter plot
fig = px.scatter(
    df_revenue,
    x='critic_score',
    y='revenue_proxy',
    color='primary_genre',
    size='owners_midpoint',
    hover_data=['title', 'price', 'owners_midpoint'],
    title=f'💰 Critic Score vs Revenue Potential<br><sub>Correlation: {corr_rev:.3f} (p={p_value_rev:.4f})</sub>',
    labels={'critic_score': 'Critic Score', 'revenue_proxy': 'Revenue Proxy ($)'},
    trendline='ols',
    trendline_scope='overall',
    height=600
)

# Add quadrant lines
fig.add_vline(x=median_score, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=median_revenue, line_dash="dash", line_color="gray", opacity=0.5)

# Add quadrant labels
fig.add_annotation(x=median_score + 1, y=median_revenue * 10, text="✅ Success Zone",
                  showarrow=False, bgcolor="lightgreen", opacity=0.7)
fig.add_annotation(x=median_score - 1, y=median_revenue * 10, text="⚠️ Hidden Gems",
                  showarrow=False, bgcolor="lightyellow", opacity=0.7)
fig.add_annotation(x=median_score + 1, y=median_revenue / 10, text="❓ Underperformers",
                  showarrow=False, bgcolor="lightcoral", opacity=0.7)

fig.update_yaxes(type='log', title='Revenue Proxy (log scale)')
fig.update_layout(
    showlegend=True,
    legend=dict(title='Primary Genre', orientation='v', x=1.02, y=1)
)

fig.show()

print(f"\n💡 Key Insight: Correlation = {corr_rev:.3f}")
if corr_rev > 0.3:
    print("✅ Strong relationship! Better reviews = Higher revenue potential")
elif corr_rev > 0.1:
    print("⚠️ Moderate relationship. Reviews help but other factors matter")
else:
    print("❌ Weak relationship. Marketing/franchise power may dominate")


💡 Key Insight: Correlation = 0.290
⚠️ Moderate relationship. Reviews help but other factors matter


## 4. Completion Behavior: Do Critics Find Quality?

**Question:** Do well-reviewed games have more committed players who finish the game?

**What to look for:**
- Higher completion rate with better scores = Critics identify quality ✅
- Shows if reviews lead to satisfied players (word-of-mouth effect)

In [7]:
# Filter data with valid completion metrics
df_completion = df_analysis[
    (df_analysis['main_story'] > 0) & 
    (df_analysis['median_playtime'] > 0)
].copy()

# Calculate completion ratio (how much of playtime is spent on main story)
df_completion['completion_ratio'] = df_completion['main_story'] / df_completion['median_playtime']

# Calculate correlation
corr_comp, p_value_comp = stats.pearsonr(df_completion['critic_score'], df_completion['completion_ratio'])

# Create scatter plot
fig = px.scatter(
    df_completion,
    x='critic_score',
    y='completion_ratio',
    color='primary_genre',
    size='owners_midpoint',
    hover_data=['title', 'main_story', 'median_playtime'],
    title=f'🏆 Critic Score vs Completion Commitment<br><sub>Correlation: {corr_comp:.3f} (p={p_value_comp:.4f})</sub>',
    labels={
        'critic_score': 'Critic Score',
        'completion_ratio': 'Completion Ratio (Main Story Time / Total Playtime)'
    },
    trendline='ols',
    trendline_scope='overall',
    height=600
)

fig.update_layout(
    showlegend=True,
    legend=dict(title='Primary Genre', orientation='v', x=1.02, y=1)
)

fig.show()

print(f"\n💡 Key Insight: Correlation = {corr_comp:.3f}")
if corr_comp > 0.3:
    print("✅ Strong relationship! Better reviews = More players finish the game")
elif corr_comp > 0.1:
    print("⚠️ Weak relationship. Quality doesn't guarantee completion")
else:
    print("❌ No relationship. Completion is independent of review scores")


💡 Key Insight: Correlation = -0.013
❌ No relationship. Completion is independent of review scores


## 5. Price Sensitivity Analysis

**Question:** Does price matter more than reviews? Can good reviews overcome high prices?

**What to look for:**
- Bubble size = Sales volume
- Color = Engagement level
- Position = Price vs. Quality balance
- **Key insight:** Find the sweet spot for pricing strategy

In [8]:
# Filter data with valid price and engagement metrics
df_price = df_engagement[
    (df_engagement['price'] > 0) & 
    (df_engagement['price'] < 100)  # Filter extreme prices
].copy()

# Create 3D relationship visualization
fig = px.scatter(
    df_price,
    x='price',
    y='critic_score',
    size='owners_midpoint',
    color='engagement_ratio',
    hover_data=['title', 'price', 'critic_score', 'owners_midpoint'],
    title='🎯 Price vs Quality vs Sales (sized by owners, colored by engagement)',
    labels={
        'price': 'Price ($)',
        'critic_score': 'Critic Score',
        'engagement_ratio': 'Engagement Ratio'
    },
    color_continuous_scale='RdYlGn',
    height=600
)

fig.update_layout(
    coloraxis_colorbar=dict(title="Engagement<br>Ratio")
)

fig.show()

print("\n💡 Key Insights:")
print(f"   • Average price for high-rated games (8+): ${df_price[df_price['critic_score'] >= 8]['price'].mean():.2f}")
print(f"   • Average price for low-rated games (<6): ${df_price[df_price['critic_score'] < 6]['price'].mean():.2f}")
print(f"   • Correlation (price vs score): {stats.pearsonr(df_price['price'], df_price['critic_score'])[0]:.3f}")


💡 Key Insights:
   • Average price for high-rated games (8+): $11.41
   • Average price for low-rated games (<6): $9.66
   • Correlation (price vs score): 0.158


## 6. Success Threshold Analysis

**Question:** What minimum critic score should Thomas target?

**What to look for:**
- Score brackets showing clear performance tiers
- Diminishing returns at higher score ranges
- **Key insight:** Identify the "safe zone" for commercial viability

In [10]:
# Create score brackets
df_analysis['score_bracket'] = pd.cut(
    df_analysis['critic_score'],
    bins=[0, 5, 6, 7, 8, 9, 10],
    labels=['0-5', '5-6', '6-7', '7-8', '8-9', '9-10']
)

# Calculate metrics by bracket
bracket_stats = df_analysis.groupby('score_bracket').agg({
    'owners_midpoint': 'mean',
    'engagement_ratio': 'mean',
    'title': 'count'
}).reset_index()
bracket_stats.columns = ['score_bracket', 'avg_owners', 'avg_engagement', 'game_count']

# Create dual-axis chart
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add bars for average owners
fig.add_trace(
    go.Bar(
        x=bracket_stats['score_bracket'],
        y=bracket_stats['avg_owners'],
        name='Average Owners',
        marker_color='lightblue',
        text=bracket_stats['avg_owners'].round(0),
        textposition='outside'
    ),
    secondary_y=False
)

# Add line for average engagement
fig.add_trace(
    go.Scatter(
        x=bracket_stats['score_bracket'],
        y=bracket_stats['avg_engagement'],
        name='Average Engagement Ratio',
        mode='lines+markers',
        line=dict(color='red', width=3),
        marker=dict(size=10)
    ),
    secondary_y=True
)

# Update axes
fig.update_xaxes(title_text='Critic Score Bracket')
fig.update_yaxes(title_text='Average Owners', secondary_y=False)
fig.update_yaxes(title_text='Average Engagement Ratio', secondary_y=True)

# Update layout
fig.update_layout(
    title='📊 Performance by Score Bracket',
    height=500,
    hovermode='x unified'
)

fig.show()

# Print key thresholds
print("\n💡 Success Thresholds:")
for idx, row in bracket_stats.iterrows():
    print(f"   Score {row['score_bracket']}: {row['game_count']:.0f} games, Avg {row['avg_owners']:.0f} owners")
    
threshold_7plus = df_analysis[df_analysis['critic_score'] >= 7]['owners_midpoint'].mean()
threshold_below7 = df_analysis[df_analysis['critic_score'] < 7]['owners_midpoint'].mean()
print(f"\n✅ Score ≥7: {threshold_7plus:.0f} avg owners")
print(f"❌ Score <7: {threshold_below7:.0f} avg owners")
print(f"📈 Improvement: {((threshold_7plus / threshold_below7 - 1) * 100):.1f}% more sales")


💡 Success Thresholds:
   Score 0-5: 82 games, Avg 382561 owners
   Score 5-6: 89 games, Avg 455730 owners
   Score 6-7: 197 games, Avg 848553 owners
   Score 7-8: 336 games, Avg 1061920 owners
   Score 8-9: 326 games, Avg 1584571 owners
   Score 9-10: 76 games, Avg 4584605 owners

✅ Score ≥7: 1628268 avg owners
❌ Score <7: 475847 avg owners
📈 Improvement: 242.2% more sales


## 7. Genre-Specific Critic Impact

**Question:** Does Thomas's game genre benefit more/less from critic reviews?

**What to look for:**
- Strong correlations in certain genres = Critics matter more there ✅
- Weak correlations = Marketing/franchise power dominates
- **Key insight:** Genre-specific strategy for review copies

In [11]:
# Get top 6 genres by game count
top_genres = df_analysis['primary_genre'].value_counts().head(6).index.tolist()
df_top_genres = df_analysis[df_analysis['primary_genre'].isin(top_genres)].copy()

# Calculate correlation by genre
genre_correlations = []
for genre in top_genres:
    genre_data = df_top_genres[df_top_genres['primary_genre'] == genre]
    if len(genre_data) > 5:  # Need enough data points
        corr, p_val = stats.pearsonr(genre_data['critic_score'], genre_data['owners_midpoint'])
        genre_correlations.append({
            'genre': genre,
            'correlation': corr,
            'p_value': p_val,
            'game_count': len(genre_data)
        })

genre_corr_df = pd.DataFrame(genre_correlations).sort_values('correlation', ascending=False)

# Create faceted scatter plots
fig = px.scatter(
    df_top_genres,
    x='critic_score',
    y='owners_midpoint',
    facet_col='primary_genre',
    facet_col_wrap=3,
    trendline='ols',
    hover_data=['title', 'critic_score', 'owners_midpoint'],
    title='🎮 Critic Impact by Genre (Top 6 Genres)',
    labels={'critic_score': 'Critic Score', 'owners_midpoint': 'Owners'},
    height=700
)

fig.update_yaxes(type='log')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

# Print correlation summary
print("\n💡 Critic Impact by Genre (Correlation with Sales):")
print("="*60)
for _, row in genre_corr_df.iterrows():
    strength = "🔥 STRONG" if row['correlation'] > 0.4 else "⚡ MODERATE" if row['correlation'] > 0.2 else "⚠️ WEAK"
    print(f"{strength:15} | {row['genre']:20} | r={row['correlation']:.3f} ({row['game_count']} games)")
    
print("\n✅ Reviews matter MOST for:", genre_corr_df.iloc[0]['genre'])
print("❌ Reviews matter LEAST for:", genre_corr_df.iloc[-1]['genre'])


💡 Critic Impact by Genre (Correlation with Sales):
⚡ MODERATE      | Puzzle               | r=0.347 (49 games)
⚡ MODERATE      | Strategy             | r=0.323 (209 games)
⚡ MODERATE      | Action               | r=0.276 (255 games)
⚠️ WEAK         | RPG                  | r=0.178 (96 games)
⚠️ WEAK         | Shooter              | r=0.176 (195 games)
⚠️ WEAK         | Adventure            | r=0.142 (133 games)

✅ Reviews matter MOST for: Puzzle
❌ Reviews matter LEAST for: Adventure


## 8. Risk Assessment: What If You Get a Bad Review?

**Question:** How much sales risk from low scores?

**What to look for:**
- Variance in lower score brackets = High risk ⚠️
- Tight distribution in high brackets = Predictable success ✅
- **Key insight:** Understand downside risk before sending review copies

In [12]:
# Create violin plot showing distribution by score bracket
fig = px.violin(
    df_analysis,
    x='score_bracket',
    y='owners_midpoint',
    box=True,
    points='outliers',
    title='📉 Sales Distribution by Critic Score (Risk Analysis)',
    labels={'score_bracket': 'Critic Score Bracket', 'owners_midpoint': 'Owners'},
    color='score_bracket',
    height=600
)

fig.update_yaxes(type='log', title='Owners (log scale)')
fig.update_layout(showlegend=False)

fig.show()

# Calculate risk metrics
print("\n💡 Risk Analysis by Score Bracket:")
print("="*70)
for bracket in ['0-5', '5-6', '6-7', '7-8', '8-9', '9-10']:
    bracket_data = df_analysis[df_analysis['score_bracket'] == bracket]['owners_midpoint']
    if len(bracket_data) > 0:
        print(f"\nScore {bracket}:")
        print(f"   📊 Median:  {bracket_data.median():>10,.0f} owners")
        print(f"   📈 Mean:    {bracket_data.mean():>10,.0f} owners")
        print(f"   📉 Std Dev: {bracket_data.std():>10,.0f} (Variance)")
        print(f"   🎮 Games:   {len(bracket_data):>10} total")
        
# Calculate downside risk
low_score_avg = df_analysis[df_analysis['critic_score'] < 6]['owners_midpoint'].mean()
high_score_avg = df_analysis[df_analysis['critic_score'] >= 8]['owners_midpoint'].mean()

print("\n" + "="*70)
print(f"\n⚠️  DOWNSIDE RISK (Score < 6): {low_score_avg:,.0f} avg owners")
print(f"✅ UPSIDE POTENTIAL (Score ≥ 8): {high_score_avg:,.0f} avg owners")
print(f"📊 Risk/Reward Ratio: {high_score_avg / low_score_avg:.1f}x")


💡 Risk Analysis by Score Bracket:

Score 0-5:
   📊 Median:      75,000 owners
   📈 Mean:       382,561 owners
   📉 Std Dev:    950,112 (Variance)
   🎮 Games:           82 total

Score 5-6:
   📊 Median:     150,000 owners
   📈 Mean:       455,730 owners
   📉 Std Dev:    728,627 (Variance)
   🎮 Games:           89 total

Score 6-7:
   📊 Median:     150,000 owners
   📈 Mean:       848,553 owners
   📉 Std Dev:  2,879,627 (Variance)
   🎮 Games:          197 total

Score 7-8:
   📊 Median:     350,000 owners
   📈 Mean:     1,061,920 owners
   📉 Std Dev:  4,354,754 (Variance)
   🎮 Games:          336 total

Score 8-9:
   📊 Median:     750,000 owners
   📈 Mean:     1,584,571 owners
   📉 Std Dev:  2,770,704 (Variance)
   🎮 Games:          326 total

Score 9-10:
   📊 Median:   1,500,000 owners
   📈 Mean:     4,584,605 owners
   📉 Std Dev: 17,259,051 (Variance)
   🎮 Games:           76 total


⚠️  DOWNSIDE RISK (Score < 6): 458,633 avg owners
✅ UPSIDE POTENTIAL (Score ≥ 8): 2,015,396 avg owners
📊

---

# 🎯 Final Recommendation: Should Thomas Send Review Copies?

## Summary Dashboard

Let's aggregate all findings to answer the core business question.

In [13]:
# Calculate all key metrics for summary
summary_metrics = {
    'Sales Correlation': stats.pearsonr(df_analysis['critic_score'], df_analysis['owners_midpoint'])[0],
    'Engagement Correlation': corr_eng,
    'Revenue Correlation': corr_rev,
    'High Score Avg Sales (≥8)': df_analysis[df_analysis['critic_score'] >= 8]['owners_midpoint'].mean(),
    'Low Score Avg Sales (<6)': df_analysis[df_analysis['critic_score'] < 6]['owners_midpoint'].mean(),
    'Score 7+ Sales Boost': ((df_analysis[df_analysis['critic_score'] >= 7]['owners_midpoint'].mean() / 
                               df_analysis[df_analysis['critic_score'] < 7]['owners_midpoint'].mean() - 1) * 100)
}

# Create summary visualization
fig = go.Figure()

# Add correlation bars
correlations = [summary_metrics['Sales Correlation'], 
                summary_metrics['Engagement Correlation'],
                summary_metrics['Revenue Correlation']]
labels = ['Sales<br>Impact', 'Engagement<br>Impact', 'Revenue<br>Impact']
colors = ['green' if c > 0.3 else 'orange' if c > 0.15 else 'red' for c in correlations]

fig.add_trace(go.Bar(
    x=labels,
    y=correlations,
    text=[f'{c:.3f}' for c in correlations],
    textposition='outside',
    marker_color=colors,
    name='Correlation Strength'
))

fig.add_hline(y=0.3, line_dash="dash", line_color="green", 
              annotation_text="Strong (>0.3)", annotation_position="right")
fig.add_hline(y=0.15, line_dash="dash", line_color="orange",
              annotation_text="Moderate (>0.15)", annotation_position="right")

fig.update_layout(
    title='📊 Critic Review Impact Score Card',
    yaxis_title='Correlation Coefficient',
    height=500,
    showlegend=False
)

fig.show()

# Print decision matrix
print("\n" + "="*70)
print("🎯 DECISION MATRIX FOR THOMAS")
print("="*70)
print(f"\n📈 Critic Score → Sales Correlation:      {summary_metrics['Sales Correlation']:>6.3f}")
print(f"🎮 Critic Score → Engagement Correlation: {summary_metrics['Engagement Correlation']:>6.3f}")
print(f"💰 Critic Score → Revenue Correlation:    {summary_metrics['Revenue Correlation']:>6.3f}")
print(f"\n✅ High Score (≥8) Avg Sales:  {summary_metrics['High Score Avg Sales (≥8)']:>10,.0f} owners")
print(f"❌ Low Score (<6) Avg Sales:   {summary_metrics['Low Score Avg Sales (<6)']:>10,.0f} owners")
print(f"📊 Sales Boost at Score 7+:    {summary_metrics['Score 7+ Sales Boost']:>9.1f}%")

print("\n" + "="*70)
if summary_metrics['Sales Correlation'] > 0.3:
    print("✅ RECOMMENDATION: SEND REVIEW COPIES")
    print("   Critics have STRONG influence on sales & engagement")
    print("   If confident in game quality (score ≥7), reviews will boost sales")
elif summary_metrics['Sales Correlation'] > 0.15:
    print("⚠️  RECOMMENDATION: CONDITIONAL - ASSESS GAME QUALITY FIRST")
    print("   Critics have MODERATE influence on sales")
    print("   Only send if expecting score ≥7, otherwise focus on marketing")
else:
    print("❌ RECOMMENDATION: FOCUS ON ALTERNATIVE MARKETING")
    print("   Critics have WEAK influence on sales in this data")
    print("   Consider influencer marketing, social media, or other channels")
print("="*70)


🎯 DECISION MATRIX FOR THOMAS

📈 Critic Score → Sales Correlation:       0.130
🎮 Critic Score → Engagement Correlation:  0.053
💰 Critic Score → Revenue Correlation:     0.290

✅ High Score (≥8) Avg Sales:   2,015,396 owners
❌ Low Score (<6) Avg Sales:      458,633 owners
📊 Sales Boost at Score 7+:        242.2%

❌ RECOMMENDATION: FOCUS ON ALTERNATIVE MARKETING
   Critics have WEAK influence on sales in this data
   Consider influencer marketing, social media, or other channels


## Key Takeaways for Thomas

### ✅ **When to Send Review Copies:**
1. **If your game is high quality** (expecting score ≥7)
   - Data shows significant sales boost for well-reviewed games
   - Creates engagement and word-of-mouth effect

2. **If your genre benefits from critics**
   - Check genre-specific analysis above
   - Some genres show stronger critic influence

3. **If you have a reasonable price point**
   - High-quality + fair price = best outcome
   - Critics help justify premium pricing

### ❌ **When NOT to Send Review Copies:**
1. **If uncertain about quality** (risk of score <6)
   - Low scores significantly hurt sales
   - Better to rely on marketing instead

2. **If in a marketing-driven genre**
   - Action/Casual games may rely more on influencers
   - Check your specific genre correlation

### **Strategic Recommendations:**
1. **Internal quality check first** - Get beta feedback before critics
2. **Target critics in your genre** - Not all critics are equal
3. **Prepare for engagement** - Good reviews = engaged players = word-of-mouth
4. **Price strategically** - Quality + fair price = maximum ROI

---

### **Next Steps for Thomas:**
- [ ] Check where your game likely falls in score brackets (beta testing)
- [ ] Identify your primary genre and review its critic correlation
- [ ] Calculate expected ROI based on score projections
- [ ] Decide: Review copies vs. alternative marketing spend